# F1 Visor · Image collection workflow

Source: the original `f1_image_collect.ipynb`, reorganized around `collect_images.py`. This notebook searches for candidate URLs and downloads a reviewed list. It does not assign training labels. See `docs/COLLECTION.md`.

Install `requirements-collect.txt` in the active kernel environment and install Microsoft Edge for the optional search step.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "f1-visor")
                if (p / "f1visor").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("Open this notebook from the f1-visor project")
print("Project:", PROJECT)


In [ ]:
CANDIDATES = PROJECT / "data" / "candidates.jsonl"
DOWNLOADS = PROJECT / "data" / "unreviewed"
QUERIES = ["F1 helmet cam view", "Formula 1 cockpit view helmet"]


## 1. Discover candidates
This cell contacts Bing through Edge. You can instead create a JSONL list of permitted image URLs yourself and skip search.

In [ ]:
command = [sys.executable, str(PROJECT / "collect_images.py"), "search", "--output", str(CANDIDATES)]
for query in QUERIES:
    command.extend(["--query", query])
subprocess.run(command, cwd=PROJECT, check=True)


## 2. Inspect sources and download
Review the source pages and permitted uses in the candidates file before executing this cell. Downloads are resumed and logged; they are not automatically added to training.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / "collect_images.py"), "download", "--input", str(CANDIDATES), "--output", str(DOWNLOADS)], cwd=PROJECT, check=True)

## 3. Label and check coverage
Select images from the camera viewpoint the prototype will use. F1 scenery alone is not a stain example. Put reviewed clear views in `data/raw/clean/` and obscured views in `data/raw/stain/`. Include real mud/water/glare examples on the prototype, record their source and capture session, then open `training.ipynb`.